# Cursul 5. Integrare, baze de date și agregare
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

În C4 am curățat sursele și am făcut tabelul regiunilor. Azi le aducem pe toate într-o **bază de date**: un singur loc, comun echipei, interogat cu **SQL**. Acolo facem două lucruri, în SQL și în pandas:

- **integrarea**: unim tabelele după cheile lor, ca să ajungem la perechi de regiuni și de țări (`join` în SQL, `merge` în pandas);
- **agregarea**: trecem de la multe rânduri la puține, un rezumat pe grup (`group by` în SQL, `groupby` în pandas).

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Baza de date SQLite și conexiunea | 5 min |
| 2 | Tabelele ca entități, cu chei, încărcate în bază | 10 min |
| 3 | Primele interogări: `select`, `where`, `order by` | 10 min |
| 4 | Integrarea în SQL: `join` | 15 min |
| 5 | Integrarea în pandas: perechile de regiuni | 10 min |
| 6 | Agregarea în SQL: `group by` | 15 min |
| 7 | Agregarea în pandas: `groupby` | 10 min |
| 8 | Perechile de țări (rulăm împreună) | 10 min |

## 1. Baza de date și conexiunea
Baza de date a cursului este **SQLite**: un singur fișier, `data/dsw.db`, pe calculatorul tău. Python o deschide fără server, fără cont și fără parolă, iar fișierul se creează singur la prima scriere.

Legătura dintre pandas și bază o face **SQLAlchemy**, printr-o adresă (*URL*). Pentru SQLite, adresa este calea fișierului. Pentru o bază PostgreSQL de pe un server, de exemplu pe Neon, se schimbă doar adresa: interogările și restul codului rămân aceleași.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/dsw.db")    # baza de date: fișierul data/dsw.db, creat la prima scriere

**Verificarea conexiunii**: cea mai simplă interogare cere bazei să răspundă cu numărul 1. `pd.read_sql` trimite interogarea și întoarce răspunsul ca tabel.

In [ ]:
pd.read_sql("select 1 as conexiunea", engine)

Dacă apare o eroare, verifică două lucruri: caietul stă în `curs/`, lângă dosarul `data/`, iar kernelul este `.venv`.

## 2. Tabelele ca entități, cu chei
O bază de date ține fiecare lucru despre care avem date (o **entitate**) într-un tabel separat: regiunile într-un tabel, perechile de regiuni în altul, migrația în altul. Fiecare tabel are o **cheie**: coloanele care identifică un rând, ca în C4. Tabelele se leagă între ele prin coloane comune: `regiune_1` din `sci` arată spre `regiune` din `regiuni`.

| Tabelul (entitatea) | Un rând | Cheia | Se leagă de |
|---|---|---|---|
| `regiuni` | o regiune NUTS 2, cu indicatorii ei | `regiune` | |
| `sci` | o pereche de regiuni, în ambele ordini | `regiune_1` + `regiune_2` | `regiuni`, prin `regiune_1` și `regiune_2` |
| `migratie` | o țară de rezidență și o cetățenie | `tara_rezidenta` + `cetatenie` | țările din perechi |
| `zboruri` | o pereche de țări | `tara_1` + `tara_2` | țările din perechi |

Le citim din depozitul cursului (tabelele salvate de caietele 3 și 4), ca toți să avem aceleași date, și le încărcăm în bază cu `to_sql`. `if_exists="replace"` înlocuiește tabelul dacă există deja, deci celula se poate rula din nou.

In [ ]:
DATE = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/processed/"   # tabelele din C3 și C4

regiuni = pd.read_csv(DATE + "regiuni_nuts2.csv")
sci = pd.read_csv(DATE + "sci_regiuni.csv")
migratie = pd.read_csv(DATE + "migratie_curata.csv", keep_default_na=False, na_values=[""])   # NA = Namibia
zboruri = pd.read_csv(DATE + "zboruri_curate.csv")

regiuni.to_sql("regiuni", engine, if_exists="replace", index=False)
sci.to_sql("sci", engine, if_exists="replace", index=False)
migratie.to_sql("migratie", engine, if_exists="replace", index=False)
zboruri.to_sql("zboruri", engine, if_exists="replace", index=False)   # afișează câte rânduri a scris

Ce tabele are acum baza? Baza își ține singură evidența, într-un catalog pe care îl putem interoga ca pe orice tabel.

In [ ]:
pd.read_sql("select name from sqlite_master where type = 'table'", engine)

Fișierul `data/dsw.db` apare în VS Code, lângă CSV-uri: toate tabelele stau acum într-un singur fișier.

## 3. Primele interogări: `select`, `where`, `order by`
O interogare SQL se citește aproape ca o propoziție:

| Cuvântul | Ce face | În pandas |
|---|---|---|
| `select` | ce coloane | `tabel[["a", "b"]]` |
| `from` | din ce tabel | |
| `where` | ce rânduri | un filtru |
| `order by ... desc` | în ce ordine | `sort_values(..., ascending=False)` |
| `limit` | câte rânduri | `head()` |

Textul din SQL se scrie între apostrofuri: `'RO'`. Interogările lungi se scriu între trei ghilimele, pe mai multe rânduri.

In [ ]:
pd.read_sql("""
    select regiune, denumire, tara, populatie
    from regiuni
    order by populatie desc
    limit 5
""", engine)

**Exercițiu.** Regiunile din România (`tara = 'RO'`), cu denumirea și PIB-ul pe locuitor, de la cel mai mare PIB la cel mai mic.

In [ ]:
pd.read_sql("""
    -- select: regiune, denumire, pib_locuitor
    -- from: regiuni
    -- where: doar România
    -- order by: pib_locuitor, descrescător
""", engine)

## 4. Integrarea în SQL: `join`
În C4 am văzut că `sci` are fiecare pereche de două ori, A-B și B-A, plus fiecare regiune cu ea însăși. Păstrăm o singură ordine, cea în care primul cod e mai mic alfabetic:

In [ ]:
pd.read_sql("select count(*) as perechi from sci where regiune_1 < regiune_2", engine)

44.253 de perechi, ca în C4. `join` face în SQL ce face `merge` în pandas: `left join regiuni r on p.regiune_1 = r.regiune` caută, pentru fiecare pereche, rândul din `regiuni` cu același cod și îi adaugă coloanele. `p` și `r` sunt nume scurte pentru cele două tabele.

In [ ]:
pd.read_sql("""
    select p.regiune_1, p.regiune_2, p.sci, r.pib_locuitor
    from sci p
    left join regiuni r on p.regiune_1 = r.regiune
    where p.regiune_1 < p.regiune_2
    limit 5
""", engine)

**Regula integrării, și în SQL:** după unire se numără rândurile. Cheia din `regiuni` este unică, deci fiecare pereche găsește cel mult un rând, iar numărul nu se schimbă.

**Exercițiu.** Câte rânduri dă unirea de mai sus? Aceeași interogare, cu `count(*)` în loc de coloane și fără `limit`.

In [ ]:
pd.read_sql("""
    -- select: câte rânduri, cu count(*)
    -- from: sci p, cu left join regiuni r pe regiune_1
    -- where: o singură ordine a perechii
""", engine)

Tot 44.253: niciun rând pierdut, niciunul înmulțit.

## 5. Integrarea în pandas: perechile de regiuni
Aceeași unire, în pandas, pentru ambele capete și cu toți cei șapte indicatori. `add_suffix("_1")` pune `_1` după fiecare nume de coloană, deci `regiune` devine `regiune_1`, exact coloana după care se face unirea.

In [ ]:
perechi = sci[sci["regiune_1"] < sci["regiune_2"]]          # o singură ordine a fiecărei perechi
atribute = regiuni.drop(columns=["tara", "denumire"])         # codul regiunii și cei șapte indicatori

perechi = perechi.merge(atribute.add_suffix("_1"), on="regiune_1", how="left")   # capătul 1
perechi = perechi.merge(atribute.add_suffix("_2"), on="regiune_2", how="left")   # capătul 2
len(perechi)                                                  # tot 44.253

Un rând pe pereche, cu indicatorii ambelor capete. Îl punem și pe el în bază, ca tabel nou:

In [ ]:
perechi.to_sql("perechi_regiuni", engine, if_exists="replace", index=False)
perechi.to_csv("data/processed/perechi_regiuni.csv", index=False)   # și ca fișier, pentru C6 și C7
perechi.head(3)

**SQL sau pandas pentru integrare?** Același rezultat. SQL lasă baza să facă munca și aduce doar ce cerem, ceea ce contează când tabelele sunt mari sau stau pe un server. Pandas e mai comod când urmează alți pași în Python. În proiect, tabelele finale se fac o singură dată, într-un singur caiet, și se pun în bază; celelalte caiete doar le citesc.

## 6. Agregarea în SQL: `group by`
`group by` împarte rândurile în grupuri și calculează câte un rezultat pe grup: `count(*)` (câte rânduri), `avg` (media), `sum` (totalul), `min`, `max`. Orice coloană din `select` care nu e calculată trebuie să apară în `group by`.

Prima întrebare: câte regiuni NUTS 2 are fiecare țară și câți locuitori au împreună?

In [ ]:
pd.read_sql("""
    select tara, count(*) as regiuni, sum(populatie) as locuitori
    from regiuni
    group by tara
    order by locuitori desc
""", engine)

Un rând pe țară: din 298 de regiuni am ajuns la câteva zeci de țări. Asta e agregarea. **Interpretează:** unele țări au o singură regiune NUTS 2. De ce contează asta când comparăm regiuni?

A doua întrebare pregătește C6: **sunt regiunile din aceeași țară mai legate între ele?** Grupăm perechile după un `case`: `case when ... then ... else ... end` dă o valoare după o condiție, ca un „dacă”.

In [ ]:
pd.read_sql("""
    select case when tara_1 = tara_2 then 'aceeasi tara' else 'tari diferite' end as tip,
           count(*) as perechi,
           avg(sci) as sci_mediu
    from perechi_regiuni
    group by tip
""", engine)

**Interpretează:** de câte ori e mai mare SCI-ul mediu în aceeași țară? Ce altceva diferă între cele două grupuri, în afară de graniță? (Indiciu: distanța. O verificăm în C6.)

**Exercițiu.** SCI-ul mediu al perechilor din interiorul fiecărei țări (`tara_1 = tara_2`), de la cel mai mare la cel mai mic.

In [ ]:
pd.read_sql("""
    -- select: tara_1 ca tara, câte perechi (count), SCI mediu (avg)
    -- from: perechi_regiuni
    -- where: aceeași țară la ambele capete
    -- group by: tara_1
    -- order by: SCI mediu, descrescător
""", engine)

## 7. Agregarea în pandas: `groupby`
`groupby` din pandas este `group by` din SQL: se alege coloana după care se grupează, coloana care se calculează și funcțiile (`count`, `sum`, `mean`).

In [ ]:
pe_tara = regiuni.groupby("tara")["populatie"].agg(["count", "sum"])   # câte regiuni, câți locuitori
pe_tara.sort_values("sum", ascending=False).head()

Același rezultat ca în SQL.

**Exercițiu.** PIB-ul mediu pe locuitor al regiunilor fiecărei țări, de la cel mai mare la cel mai mic.

In [ ]:
# groupby pe tara, coloana pib_locuitor, media (mean)
# apoi sort_values descrescător și head()

**Atenție la medii:** media PIB-ului regiunilor unei țări nu e PIB-ul țării. Regiunile au populații diferite, iar o regiune mică cântărește la fel de mult ca una mare. Când agregăm, ne întrebăm întâi ce reprezintă un rând.

## 8. Perechile de țări (rulăm împreună)
> Această secțiune o rulăm împreună: integrarea și agregarea, pe un tabel nou.

Un rând va fi o pereche de țări, cu SCI, migrația în **ambele sensuri** și zborurile. Întâi o agregare: nu toate țările trimit la Eurostat migranții pe fiecare cetățenie. Numărăm, pentru fiecare țară de rezidență, câte cetățenii au o valoare; `count` numără doar valorile prezente, ca în SQL `count(persoane)`.

In [ ]:
acoperire = migratie.groupby("tara_rezidenta")["persoane"].count()   # câte cetățenii cu valoare, pe țară
raportoare = acoperire[acoperire >= 20].index                       # țările care raportează cel puțin 20
len(raportoare)

SCI pe țări e în `country.csv`, din C1. Păstrăm o singură ordine a fiecărei perechi și doar țările care raportează.

In [ ]:
tari = pd.read_csv("https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/country.csv",
                   keep_default_na=False, na_values=[""])                         # NA = Namibia
tari = tari.rename(columns={"user_country": "tara_1", "friend_country": "tara_2", "scaled_sci": "sci"})
tari = tari[["tara_1", "tara_2", "sci"]]

ambele = tari["tara_1"].isin(raportoare) & tari["tara_2"].isin(raportoare)       # ambele țări raportează
tari = tari[(tari["tara_1"] < tari["tara_2"]) & ambele]                          # și o singură ordine
len(tari)

Migrația are direcție: românii din Italia nu sunt același lucru cu italienii din România. O lipim de două ori, câte o dată pentru fiecare sens, redenumind coloanele ca să se potrivească cu cheile din `tari`. Apoi adunăm cele două sensuri: dacă unul lipsește, suma rămâne lipsă. Zborurile le lipim o dată.

In [ ]:
m = migratie.rename(columns={"cetatenie": "tara_1", "tara_rezidenta": "tara_2", "persoane": "din_1_in_2"})
tari = tari.merge(m, on=["tara_1", "tara_2"], how="left")      # sensul 1: cetățenii țării 1 din țara 2

m = migratie.rename(columns={"cetatenie": "tara_2", "tara_rezidenta": "tara_1", "persoane": "din_2_in_1"})
tari = tari.merge(m, on=["tara_1", "tara_2"], how="left")      # sensul 2: cetățenii țării 2 din țara 1

tari["migranti"] = tari["din_1_in_2"] + tari["din_2_in_1"]    # o valoare lipsă dă suma lipsă
len(tari)

Zborurile se adaugă o singură dată, apoi tabelul intră în bază.

In [ ]:
tari = tari.merge(zboruri, on=["tara_1", "tara_2"], how="left")   # pasagerii raportați de țara 1
tari.to_sql("perechi_tari", engine, if_exists="replace", index=False)
tari.to_csv("data/processed/perechi_tari.csv", index=False)   # și ca fișier, pentru C6 și C7
len(tari)

Perechea Italia-România, direct din bază:

In [ ]:
pd.read_sql("""
    select tara_1, tara_2, sci, din_1_in_2, din_2_in_1, migranti, pasageri
    from perechi_tari
    where tara_1 = 'IT' and tara_2 = 'RO'
""", engine)

**Validarea finală.** Cele trei tabele de analiză, cu regulile lor: câte rânduri, cheia unică, SCI complet.

In [ ]:
print(len(regiuni), regiuni.duplicated("regiune").sum())                    # rânduri, chei repetate
print(len(perechi), perechi.duplicated(["regiune_1", "regiune_2"]).sum())
print(len(tari), tari.duplicated(["tara_1", "tara_2"]).sum())

## Echipa: baza proiectului
Baza echipei este un fișier din depozit, `data/processed/proiect.db`, ca toți să lucreze pe aceleași date:
1. Caietul 03 al echipei creează baza și încarcă tabelele finale cu `to_sql`. Caietele stau în `notebooks/`, deci adresa urcă un nivel: `create_engine("sqlite:///../data/processed/proiect.db")`.
2. Fișierul se urcă pe GitHub împreună cu caietul; colegii îl primesc cu `git pull`.
3. Caietele 04 și 05 doar citesc din bază, cu `pd.read_sql`. **Doar caietul 03 scrie în ea**: Git nu poate uni două versiuni diferite ale aceluiași fișier de bază de date.

**Exercițiul echipei (dacă rămâne timp).** O interogare cu `group by` despre coridorul vostru: câte perechi de regiuni are coridorul și care e SCI-ul lor mediu.

In [ ]:
pd.read_sql("""
    -- select: tara_1, tara_2, câte perechi, SCI mediu
    -- from: perechi_regiuni
    -- where: cele două țări ale coridorului, la capete diferite (de exemplu 'IT' și 'RO')
    -- group by: tara_1, tara_2
""", engine)

## Ce ducem mai departe
Cele trei tabele de analiză (`regiuni`, `perechi_regiuni`, `perechi_tari`) sunt acum în baza de date. În C6 le citim de acolo, cu `pd.read_sql`, pentru analiza exploratorie.

**Poți explica fără cod?** Ce este o entitate și ce este o cheie? Ce face un `join` și prin ce seamănă cu `merge`? De ce numărăm rândurile după fiecare unire? Ce înseamnă să agregăm? De ce media pe regiuni a unei țări nu e valoarea țării?

### Documentație
- [SQLite: când se folosește o bază într-un fișier](https://www.sqlite.org/whentouse.html)
- [SQLAlchemy: `create_engine` și adresele bazelor de date](https://docs.sqlalchemy.org/en/21/core/engines.html)
- [pandas: `read_sql` și `to_sql`](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html)
- [pandas: `groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [PostgreSQL: `select`, `join`, `group by`](https://www.postgresql.org/docs/current/sql-select.html)
- [PostgreSQL tutorial: unirea tabelelor](https://www.postgresql.org/docs/current/tutorial-join.html)